In [1]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

import os
os.environ["HF_TOKEN"] = HF_TOKEN

import os
if os.path.isdir("repo"):
    print("repo/ already exists -- refreshing to a clean phase2 checkout")
    !cd repo && git fetch origin phase2 && git checkout phase2 && git reset --hard origin/phase2
else:
    !git clone -b phase2 https://{GITHUB_TOKEN}@github.com/CL3-410/individual-project-ManyaJain111.git repo

Cloning into 'repo'...


remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (351/351), done.


remote: Compressing objects: 100% (275/275), done.


remote: Total 351 (delta 161), reused 205 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (351/351), 19.50 MiB | 21.11 MiB/s, done.


Resolving deltas: 100% (161/161), done.


In [2]:
!pip install -q huggingface_hub datasets sacrebleu rouge_score seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.6 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.6 MB/s eta 0:00:00


In [3]:
train_py_path = "repo/phase2/src/train.py"
with open(train_py_path) as f:
    src = f.read()

old_final_save = '''    torch.save({
        'step': step,
        'model_state_dict': get_model_state_dict(model),
        'config': config
    }, os.path.join(args.output_dir, "model_final.pt"))'''
new_final_save = '''    torch.save({
        'step': step,
        'model_state_dict': get_model_state_dict(model),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'config': config
    }, os.path.join(args.output_dir, "model_final.pt"))'''
if "'optimizer_state_dict': optimizer.state_dict()" in src:
    print("ℹ️ final-save already includes optimizer/scheduler state — skipping (idempotent)")
elif old_final_save in src:
    src = src.replace(old_final_save, new_final_save)
else:
    print("⚠️ final save block not found — may already be patched, or repo changed")

with open(train_py_path, "w") as f:
    f.write(src)

print("✅ train.py patch complete")

ℹ️ final-save already includes optimizer/scheduler state — skipping (idempotent)
✅ train.py patch complete


In [4]:
import shutil

print('--- disk usage ---')
!df -h /kaggle/working
print()
print('--- biggest items in /kaggle/working ---')
!du -sh /kaggle/working/* 2>/dev/null | sort -rh | head -20

_extract_dir = '/kaggle/working/lma_extracted/nepali-lma-corpus'
if os.path.exists(_extract_dir) and not os.path.exists(f'{_extract_dir}/.done'):
    print(f'\nRemoving incomplete extraction at {_extract_dir} (no .done marker -> it never finished)')
    shutil.rmtree(_extract_dir)

for _f in ['nepali_train.bin', 'nepali_val.bin', 'nepali_test.bin']:
    _p = f'/kaggle/working/{_f}'
    if os.path.exists(_p):
        print(f'Removing stale {_p}')
        os.remove(_p)

_resplit_dir = '/kaggle/working/nepali_resplit'
if os.path.exists(_resplit_dir):
    print(f'Removing stale {_resplit_dir}')
    shutil.rmtree(_resplit_dir)

--- disk usage ---
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   92M   20G   1% /kaggle/working



--- biggest items in /kaggle/working ---
92M	/kaggle/working/repo
64K	/kaggle/working/__notebook__.ipynb


In [5]:
import os, json, re, string, glob, tarfile, numpy as np
from multiprocessing import Pool, cpu_count
from huggingface_hub import hf_hub_download

EXTRACT_DIR = '/kaggle/working/lma_extracted/nepali-lma-corpus'
TOK_PATH    = 'repo/phase2/tokenizers/nepali_tokenizer_16000.json'
OUT_DIR     = '/kaggle/working'
MAX_TRAIN   = 500_000_000
CHUNK       = 5000

def _required_split_files_present(extract_dir):
    return all(glob.glob(f'{extract_dir}/**/{fname}', recursive=True)
               for fname in ('train.txt', 'val.txt', 'test.txt'))

if os.path.exists(f'{EXTRACT_DIR}/.done') and not _required_split_files_present(EXTRACT_DIR):
    print(f"[Dataset] {EXTRACT_DIR}/.done exists but train/val/test.txt are missing "
          f"(likely deleted by the dedup cell in an earlier run) -- re-extracting.")
    os.remove(f'{EXTRACT_DIR}/.done')

if not os.path.exists(f'{EXTRACT_DIR}/.done'):
    print("[Dataset] Downloading nepali_submission_phase1.tar.gz...")
    tar_path = hf_hub_download(repo_id="manyajain111/nepali-lma-corpus", filename="nepali_submission_phase1.tar.gz", repo_type="dataset")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=EXTRACT_DIR)
    open(f'{EXTRACT_DIR}/.done', 'w').close()
    print("[Dataset] Extraction done!")
else:
    print(f"[Dataset] Already extracted in {EXTRACT_DIR}")

PUNCT    = re.escape(string.punctuation + "।॥''\"\"")
WORD_PAT = re.compile(rf" ?[^{PUNCT}\s]+| ?[{PUNCT}]+", re.UNICODE)
_vocab = _merge_ranks = _b2c = _cache = _mode = None

def detect_bpe_mode(vocab):
    """
    Two training regimes produce a tokenizer json with the same
    'vocab'/'merges' shape on disk, but they are NOT interchangeable:
      - byte-level BPE (Hindi): base alphabet = the 256 raw UTF-8 bytes,
        remapped 1:1 to chr(0..255) before merges are learned.
      - char-level BPE (Nepali, per Phase-1 report): base alphabet =
        actual Unicode codepoints, with merges learned directly over them.
    Feeding a char-level tokenizer's merge table a byte-decomposed input
    (or vice versa) means the merge lookup never matches, BPE breaks
    immediately for every word, and encoding silently degrades to
    ~1 token per UTF-8 byte -- ~3x inflation for 3-byte Devanagari on top
    of losing all subword compression.
    Detect by checking whether any vocab entry contains a Devanagari
    codepoint (U+0900-U+097F) directly: byte-remapped entries can't,
    since chr(0..255) only spans Latin-1.
    """
    for tok in vocab:
        if any(0x0900 <= ord(ch) <= 0x097F for ch in tok):
            return 'char'
    return 'byte'

def init_worker(tok_path):
    global _vocab, _merge_ranks, _b2c, _cache, _mode
    with open(tok_path) as f: data = json.load(f)
    _vocab = data['vocab']
    merges = [((a, b), a + b) for a, b in data['merges']]
    _merge_ranks = {p: i for i, (p, _) in enumerate(merges)}
    _b2c = {i: chr(i) for i in range(256)}
    _mode = detect_bpe_mode(_vocab)
    _cache = {}

def _base_symbols(w):
    if _mode == 'byte':
        return [_b2c[b] for b in w.encode('utf-8')]
    return list(w)

def encode_chunk(lines):
    out = []
    for line in lines:
        line = line.strip()
        if not line: continue
        for w in WORD_PAT.findall(line):
            if w not in _cache:
                s = _base_symbols(w)
                while len(s) > 1:
                    pairs  = [(s[i], s[i+1]) for i in range(len(s)-1)]
                    ranked = [(_merge_ranks[p], i) for i, p in enumerate(pairs) if p in _merge_ranks]
                    if not ranked: break
                    _, i = min(ranked)
                    s = s[:i] + [s[i] + s[i+1]] + s[i+2:]
                _cache[w] = [_vocab.get(x, 0) for x in s]
            out.extend(_cache[w])
    return out

def tokenize_to_file(fnames, out_path, max_tok=None, base_dir=None):
    import time
    base_dir = base_dir or EXTRACT_DIR
    fpaths = []
    for f in fnames:
        matches = glob.glob(f'{base_dir}/**/{f}', recursive=True)
        if matches: fpaths.append(matches[0])
    if not fpaths:
        print(f"❌ Error: Could not find any of {fnames} in {EXTRACT_DIR}")
        return
    if os.path.exists(out_path): os.remove(out_path)
    total_tokens = 0
    t0 = time.time()
    with open(out_path, 'wb') as f_out:
        for fpath in fpaths:
            print(f'Reading {os.path.basename(fpath)}...')
            lines = open(fpath, encoding='utf-8', errors='ignore').readlines()
            print(f'  {len(lines):,} lines | {cpu_count()} CPUs')
            chunks = [lines[i:i+CHUNK] for i in range(0, len(lines), CHUNK)]
            with Pool(cpu_count(), initializer=init_worker, initargs=(TOK_PATH,)) as pool:
                for i, r in enumerate(pool.imap(encode_chunk, chunks)):
                    arr = np.array(r, dtype=np.uint16)
                    f_out.write(arr.tobytes())
                    total_tokens += len(arr)
                    if i % 20 == 0:
                        tps = total_tokens / max(time.time() - t0, 1)
                        print(f'  {i}/{len(chunks)} chunks | {total_tokens:,} tokens | {tps:,.0f} tok/s', flush=True)
                    if max_tok and total_tokens >= max_tok:
                        pool.terminate()
                        break
            if max_tok and total_tokens >= max_tok: break
    print(f'✅ Saved {total_tokens:,} tokens -> {out_path}\n')

[Dataset] Downloading nepali_submission_phase1.tar.gz...


nepali_submission_phase1.tar.gz:   0%|          | 0.00/1.72G [00:00<?, ?B/s]

/tmp/ipykernel_23/3113873609.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACT_DIR)


[Dataset] Extraction done!


In [6]:
def load_lines(fname):
    path = glob.glob(f'{EXTRACT_DIR}/**/{fname}', recursive=True)[0]
    with open(path, encoding='utf-8', errors='ignore') as f:
        lines = set(line.rstrip('\n') for line in f)
    print(f"{fname}: {len(lines):,} unique lines")
    return lines

_train_lines = load_lines('train.txt')
_val_lines   = load_lines('val.txt')
_test_lines  = load_lines('test.txt')

print("train \u2229 val:", len(_train_lines & _val_lines), f"({100*len(_train_lines & _val_lines)/len(_val_lines):.1f}% of val)")
print("train \u2229 test:", len(_train_lines & _test_lines), f"({100*len(_train_lines & _test_lines)/len(_test_lines):.1f}% of test)")
print("val \u2229 test:", len(_val_lines & _test_lines))
del _train_lines, _val_lines, _test_lines

train.txt: 3,448,616 unique lines


val.txt: 54,918 unique lines


test.txt: 54,263 unique lines
train ∩ val: 31290 (57.0% of val)
train ∩ test: 30840 (56.8% of test)
val ∩ test: 2093


In [7]:
import random

SPLIT_DIR = f'{OUT_DIR}/nepali_resplit'
os.makedirs(SPLIT_DIR, exist_ok=True)

SEED      = 42
VAL_FRAC  = 0.02
TEST_FRAC = 0.02

def find_file(fname):
    matches = glob.glob(f'{EXTRACT_DIR}/**/{fname}', recursive=True)
    return matches[0] if matches else None

seen = set()
pooled = []
for fname in ['train.txt', 'val.txt', 'test.txt']:
    fpath = find_file(fname)
    if not fpath:
        print(f'\u26a0\ufe0f {fname} not found, skipping')
        continue
    n_before = n_kept = 0
    with open(fpath, encoding='utf-8', errors='ignore') as f:
        for line in f:
            n_before += 1
            line = line.strip()
            if not line or line in seen:
                continue
            seen.add(line)
            pooled.append(line)
            n_kept += 1
    print(f'{fname}: {n_before:,} lines -> {n_kept:,} new unique lines (running total {len(pooled):,})')
    os.remove(fpath)
    print(f'  (freed disk space -- removed original {fpath})')

print(f'\nTotal unique lines after pooling all three splits: {len(pooled):,}')
del seen

random.Random(SEED).shuffle(pooled)

n_val  = int(len(pooled) * VAL_FRAC)
n_test = int(len(pooled) * TEST_FRAC)
val_lines   = pooled[:n_val]
test_lines  = pooled[n_val:n_val + n_test]
train_lines = pooled[n_val + n_test:]

print(f'New split -> train: {len(train_lines):,} lines | val: {len(val_lines):,} | test: {len(test_lines):,}')

for name, lines in [('train', train_lines), ('val', val_lines), ('test', test_lines)]:
    with open(f'{SPLIT_DIR}/{name}.txt', 'w', encoding='utf-8') as f:
        for line in lines:
            f.write(line + '\n')
print(f'\u2705 Wrote deduped, disjoint splits to {SPLIT_DIR}')

train.txt: 6,277,197 lines -> 3,448,616 new unique lines (running total 3,448,616)


  (freed disk space -- removed original /kaggle/working/lma_extracted/nepali-lma-corpus/datasets/train.txt)


val.txt: 64,410 lines -> 23,628 new unique lines (running total 3,472,244)
  (freed disk space -- removed original /kaggle/working/lma_extracted/nepali-lma-corpus/datasets/val.txt)


test.txt: 63,657 lines -> 23,269 new unique lines (running total 3,495,513)
  (freed disk space -- removed original /kaggle/working/lma_extracted/nepali-lma-corpus/datasets/test.txt)

Total unique lines after pooling all three splits: 3,495,513


New split -> train: 3,355,693 lines | val: 69,910 | test: 69,910


✅ Wrote deduped, disjoint splits to /kaggle/working/nepali_resplit


In [8]:
def load_lines_split(fname):
    with open(f'{SPLIT_DIR}/{fname}', encoding='utf-8', errors='ignore') as f:
        lines = set(line.rstrip('\n') for line in f)
    print(f"{fname}: {len(lines):,} unique lines")
    return lines

_train2 = load_lines_split('train.txt')
_val2   = load_lines_split('val.txt')
_test2  = load_lines_split('test.txt')

print("train \u2229 val:", len(_train2 & _val2))
print("train \u2229 test:", len(_train2 & _test2))
print("val \u2229 test:", len(_val2 & _test2))
del _train2, _val2, _test2

train.txt: 3,355,693 unique lines


val.txt: 69,910 unique lines


test.txt: 69,910 unique lines
train ∩ val: 0
train ∩ test: 0
val ∩ test: 0


In [9]:
with open(TOK_PATH) as f:
    _tok_data = json.load(f)
_detected_mode = detect_bpe_mode(_tok_data['vocab'])
print(f"Detected BPE base-alphabet mode for {TOK_PATH}: '{_detected_mode}'")
print("(byte-level = raw UTF-8 bytes remapped to chr(0..255); "
      "char-level = raw Unicode codepoints)")

init_worker(TOK_PATH)
with open(f'{SPLIT_DIR}/train.txt', encoding='utf-8') as f:
    _sample_lines = [next(f) for _ in range(200)]
_sample_tokens = encode_chunk(_sample_lines)
_sample_words = sum(len(WORD_PAT.findall(l.strip())) for l in _sample_lines if l.strip())
_fertility = len(_sample_tokens) / _sample_words

print(f"Sample: {len(_sample_lines)} lines | {_sample_words} words | {len(_sample_tokens)} tokens")
print(f"Fertility (tokens/word): {_fertility:.4f}  <- compare to Phase-1 report's 1.2059")

if _fertility > 2.0:
    print("\n\u26a0\ufe0f Still way above 1.2 -- detect_bpe_mode() likely guessed wrong, "
          "or the merge table itself doesn't match either hypothesis. "
          "STOP and inspect nepali_tokenizer_16000.json's 'vocab'/'merges' "
          "structure directly before re-running tokenize_to_file.")
else:
    print("\n\u2705 Fertility is in the expected range -- safe to proceed to tokenize_to_file below.")

Detected BPE base-alphabet mode for repo/phase2/tokenizers/nepali_tokenizer_16000.json: 'char'
(byte-level = raw UTF-8 bytes remapped to chr(0..255); char-level = raw Unicode codepoints)


Sample: 200 lines | 18204 words | 23406 tokens
Fertility (tokens/word): 1.2858  <- compare to Phase-1 report's 1.2059

✅ Fertility is in the expected range -- safe to proceed to tokenize_to_file below.


In [10]:
tokenize_to_file(['train.txt'], f'{OUT_DIR}/nepali_train.bin', max_tok=MAX_TRAIN, base_dir=SPLIT_DIR)
tokenize_to_file(['val.txt'], f'{OUT_DIR}/nepali_val.bin', base_dir=SPLIT_DIR)
tokenize_to_file(['test.txt'], f'{OUT_DIR}/nepali_test.bin', base_dir=SPLIT_DIR)

Reading train.txt...


  3,355,693 lines | 4 CPUs


  0/672 chunks | 598,146 tokens | 27,570 tok/s


  20/672 chunks | 13,321,774 tokens | 416,932 tok/s


  40/672 chunks | 26,170,577 tokens | 649,162 tok/s


  60/672 chunks | 38,944,717 tokens | 803,138 tok/s


... (33 outputs left out)

✅ Saved 8,870,301 tokens -> /kaggle/working/nepali_val.bin

Reading test.txt...


  69,910 lines | 4 CPUs


  0/14 chunks | 637,774 tokens | 120,259 tok/s


✅ Saved 8,957,744 tokens -> /kaggle/working/nepali_test.bin



In [11]:
!python repo/phase2/src/train.py \
    --config repo/phase2/config/model_l_config.json \
    --tokenizer repo/phase2/tokenizers/nepali_tokenizer_16000.json \
    --dataset manyajain111/nepali-lma-corpus \
    --output_dir /kaggle/working/nepali_checkpoints \
    --batch_size 16 --grad_accum 4 \
    --max_steps 61036 --save_steps 1000 --val_steps 1000

Using device: cuda


🚀 Using 2 GPUs with DataParallel!
Total parameters: 24.34 M


[Dataset] Memory-mapping /kaggle/working/nepali_train.bin...
[Dataset] 429,660,602 tokens ready.
[Dataset] Memory-mapping /kaggle/working/nepali_val.bin...
[Dataset] 8,870,301 tokens ready.
Training:   0%|                                       | 0/61036 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training:   0%|                            | 1/61036 [00:02<42:41:56,  2.52s/it]

Training:   0%|                            | 2/61036 [00:03<22:44:34,  1.34s/it]

... (11 outputs left out)

Training:  77%|▊| 47000/61036 [7:00:46<2:04:04,  1.89it/s, loss=3.6789, lr=3.99e


Step 47000: Val Loss 3.6270 | Val PPL 37.60


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 977, in save
    _save(
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1286, in _save
    zip_file.write_record(name, storage, num_bytes)
RuntimeError: basic_ios::clear: iostream error

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/kaggle/working/repo/phase2/src/train.py", line 192, in <module>
    train()
  File "/kaggle/working/repo/phase2/src/train.py", line 177, in train
    torch.save(checkpoint, os.path.join(args.output_dir, f"ckpt_{step}.pt"))
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 976, in save
    with _open_zipfile_writer(f) as opened_zipfile:
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 808, in __exit__
    self.file_like.write_end_of_file()
RuntimeError: [enforce f

Training:  77%|▊| 47000/61036 [7:00:52<2:05:41,  1.86it/s, loss=3.6789, lr=3.99e


In [ ]:
!python repo/phase2/src/evaluate.py \
    --checkpoint /kaggle/working/nepali_checkpoints/model_final.pt \
    --tokenizer repo/phase2/tokenizers/nepali_tokenizer_16000.json \
    --dataset manyajain111/nepali-lma-corpus \
    --language nepali \
    --output_dir /kaggle/working/nepali_reports